# 1D Reservoir Simulation - Single Phase Flow

This notebook demonstrates the mathematical modeling and numerical simulation of single-phase fluid flow in a 1D porous medium.

## 1. Governing Equation

The 1D diffusivity equation for a reservoir is:
$$\frac{\partial^2 p}{\partial x^2} = \frac{1}{\eta} \frac{\partial p}{\partial t}$$

Where $\eta = \frac{0.000264 k}{\phi \mu c_t}$ (in field units).

## 2. Detailed Analytical Trace (Sympy)

We derive the solution for a block saturated at $p_i$ that is suddenly produced at $x=0$ at a constant pressure $P_{wf}$.

In [ ]:
import sympy as sp
from IPython.display import display, Math

sp.init_printing()
x, t, eta, L = sp.symbols('x t eta L', real=True, positive=True)
P = sp.Function('P')(x, t)

# Governing Diffusivity Equation
pde = sp.Eq(P.diff(x, x), (1/eta) * P.diff(t))

# Separation of variables ansatz
X = sp.Function('X')(x)
Phi = sp.Function('Phi')(t)
lam = sp.symbols('lambda', real=True, positive=True)

spatial_ode = sp.Eq(X.diff(x, x) + lam**2 * X, 0)
temporal_ode = sp.Eq(Phi.diff(t) + eta * lam**2 * Phi, 0)

display(Math(f"\\text{{Spatial ODE: }} {sp.latex(spatial_ode)}"))
display(Math(f"\\text{{Temporal ODE: }} {sp.latex(temporal_ode)}"))

sol_x = sp.dsolve(spatial_ode, X)
sol_phi = sp.dsolve(temporal_ode, Phi)
display(Math(f"X(x) = {sp.latex(sol_x.rhs)}"))
display(Math(f"\\Phi(t) = {sp.latex(sol_phi.rhs)}"))

## 3. Numerical Simulation

We use the **Finite Volume Method (FVM)** for the numerical backend, as is standard in the oil and gas industry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solve_reservoir_1d(nx, L, nt, dt, eta, pi, pwf):
    dx = L / nx
    p = pi * np.ones(nx)
    for _ in range(nt):
        pn = p.copy()
        for i in range(1, nx - 1):
            p[i] = pn[i] + eta * dt / dx**2 * (pn[i+1] - 2*pn[i] + pn[i-1])
        # Left wall BC: Constant Well Pressure
        p[0] = pwf
        # Right wall BC: No Flow (d P / d x = 0)
        p[-1] = p[-2]
    return p

nx = 50
L = 1000.0
eta_val = 10.0
nt = 500
dt = 1.0
pi = 3000.0
pwf = 1000.0

p_res = solve_reservoir_1d(nx, L, nt, dt, eta_val, pi, pwf)
x_vals = np.linspace(0, L, nx)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, p_res, 'b-', label='Pressure Profile')
plt.axhline(pi, color='gray', linestyle='--', label='Initial Pressure')
plt.title("1D Reservoir Depletion Profile")
plt.xlabel("Distance from Well (ft)")
plt.ylabel("Pressure (psi)")
plt.legend()
plt.grid(True)
plt.show()